# Tarea 3 — PyTorch

**Nombre del integrante:** Giselle Esclasans

**Cédula del integrante:** 30715145

> Enunciado, rúbrica y nombres de los archivos `.pth` a entregar: ver `README.md` de esta carpeta.
> Antes del commit: *Restart & Run All* debe terminar sin errores y quedar en el menú.

## 1. Imports y constantes

In [1]:
import copy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms

NOTEBOOK_NUMBER = 3
DATA_DIR = Path("data")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

INPUT_SIZE = 784
BATCH_SIZE = 128
VALIDATION_SIZE = 5000   # taken from the training set, used by early stopping

DEFAULT_HIDDEN_LAYERS = 2
DEFAULT_NEURONS = 512
DEFAULT_EPOCHS = 10

# Optimizations available in the menu, all disabled except for the optimizer itself.
DEFAULT_OPTIONS = {
    "optimizer": "adam",
    "learning_rate": 0.001,
    "dropout": 0.0,
    "batch_norm": False,
    "weight_decay": 0.0,
    "early_stopping": False,
    "patience": 3,
}

OPTIMIZERS = {
    "adam": lambda params, lr, decay: torch.optim.Adam(params, lr=lr, weight_decay=decay),
    "adamw": lambda params, lr, decay: torch.optim.AdamW(params, lr=lr, weight_decay=decay),
    "sgd": lambda params, lr, decay: torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=decay),
    "rmsprop": lambda params, lr, decay: torch.optim.RMSprop(params, lr=lr, weight_decay=decay),
}

print(f"PyTorch {torch.__version__} | device: {DEVICE}")


PyTorch 2.15.0+cpu | device: cpu


## 2. Datos (Fashion MNIST original y clases simplificadas)

In [2]:
CLASS_NAMES = {
    "original": ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
                 "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"],
    "simplified": ["Top", "Bottom", "Footwear", "Bag"],
}

# simplified label of each original label: 0 = Top, 1 = Bottom, 2 = Footwear, 3 = Bag
SIMPLIFIED_LABELS = torch.tensor([0, 1, 0, 1, 0, 2, 0, 2, 3, 2])


def read_split(train):
    """Returns (images, labels) of one split, already converted by ToTensor."""
    dataset = datasets.FashionMNIST(DATA_DIR, train=train, download=True,
                                    transform=transforms.ToTensor())
    images = torch.stack([image for image, _ in dataset])
    return images, dataset.targets.clone()


TRAIN_IMAGES, TRAIN_LABELS = read_split(train=True)
TEST_IMAGES, TEST_LABELS = read_split(train=False)


def labels_for(labels, classification):
    return SIMPLIFIED_LABELS[labels] if classification == "simplified" else labels


def make_loaders(classification):
    """Returns the (train, validation, test) loaders of a classification."""
    full_train = TensorDataset(TRAIN_IMAGES, labels_for(TRAIN_LABELS, classification))
    test = TensorDataset(TEST_IMAGES, labels_for(TEST_LABELS, classification))

    # The split is fixed so that the validation samples never leak into training.
    sizes = [len(full_train) - VALIDATION_SIZE, VALIDATION_SIZE]
    train, validation = random_split(full_train, sizes,
                                     generator=torch.Generator().manual_seed(SEED))

    return (DataLoader(train, batch_size=BATCH_SIZE, shuffle=True),
            DataLoader(validation, batch_size=BATCH_SIZE),
            DataLoader(test, batch_size=BATCH_SIZE))


print(f"train: {len(TRAIN_IMAGES)} images | test: {len(TEST_IMAGES)} images")


train: 60000 images | test: 10000 images


## 3. Modelos (rectangular y convolucional)

In [3]:
def hidden_sizes_for(architecture, hidden_layers, neurons_per_layer, output_size):
    """Neurons of each hidden layer."""
    if architecture == "rectangular":
        return [neurons_per_layer] * hidden_layers

    # Convolutional: linear interpolation, every layer shrinks by the same amount.
    return [round(INPUT_SIZE + (output_size - INPUT_SIZE) * i / (hidden_layers + 1))
            for i in range(1, hidden_layers + 1)]


def build_model(hidden_sizes, output_size, dropout=0.0, batch_norm=False):
    """nn.Sequential with only Flatten, Linear, BatchNorm1d, ReLU and Dropout."""
    layers = [nn.Flatten()]
    previous = INPUT_SIZE

    for size in hidden_sizes:
        layers.append(nn.Linear(previous, size))
        if batch_norm:
            layers.append(nn.BatchNorm1d(size))
        layers.append(nn.ReLU())
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        previous = size

    layers.append(nn.Linear(previous, output_size))
    return nn.Sequential(*layers).to(DEVICE)


def create_network(classification="original", architecture="rectangular",
                   hidden_layers=DEFAULT_HIDDEN_LAYERS, neurons_per_layer=DEFAULT_NEURONS,
                   **options):
    """A network is its model plus the hyperparameters saved next to it."""
    torch.manual_seed(SEED)
    output_size = len(CLASS_NAMES[classification])
    hidden_sizes = hidden_sizes_for(architecture, hidden_layers, neurons_per_layer, output_size)
    optimizations = {**DEFAULT_OPTIONS, **options}

    return {
        "model": build_model(hidden_sizes, output_size,
                             optimizations["dropout"], optimizations["batch_norm"]),
        "hyperparameters": {
            "architecture": architecture,
            "hidden_layers": hidden_layers,
            "neurons_per_layer": hidden_sizes,
            "epochs": 0,
            "classification": classification,
            "optimizations": optimizations,
        },
    }


def describe(network):
    hp = network["hyperparameters"]
    sizes = [INPUT_SIZE] + hp["neurons_per_layer"] + [len(CLASS_NAMES[hp["classification"]])]
    return (f"{hp['architecture']} / {hp['classification']} | "
            f"{' -> '.join(map(str, sizes))} | {hp['epochs']} epochs")


print(describe(create_network()))
print(describe(create_network(architecture="convolutional")))


rectangular / original | 784 -> 512 -> 512 -> 10 | 0 epochs
convolutional / original | 784 -> 526 -> 268 -> 10 | 0 epochs


## 4. Entrenar, probar, guardar y cargar

In [4]:
def average_loss(model, loader, criterion):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            total += criterion(model(images), labels).item() * len(labels)
    return total / len(loader.dataset)


def train_model(network, epochs):
    """Trains the network and returns the (train, validation) loss of each epoch."""
    model = network["model"]
    hp = network["hyperparameters"]
    options = hp["optimizations"]

    train_loader, validation_loader, _ = make_loaders(hp["classification"])
    criterion = nn.CrossEntropyLoss()
    optimizer = OPTIMIZERS[options["optimizer"]](model.parameters(), options["learning_rate"],
                                                 options["weight_decay"])

    train_losses, validation_losses = [], []
    best_loss, best_state, waited = float("inf"), None, 0

    for epoch in range(1, epochs + 1):
        model.train()
        total = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(labels)

        train_losses.append(total / len(train_loader.dataset))
        validation_losses.append(average_loss(model, validation_loader, criterion))
        hp["epochs"] += 1
        print(f"epoch {epoch}/{epochs} - train loss: {train_losses[-1]:.4f}"
              f" - validation loss: {validation_losses[-1]:.4f}")

        if options["early_stopping"]:
            if validation_losses[-1] < best_loss:
                best_loss, waited = validation_losses[-1], 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                waited += 1
                if waited >= options["patience"]:
                    print(f"early stopping at epoch {epoch}")
                    break

    if best_state is not None:
        model.load_state_dict(best_state)

    return train_losses, validation_losses


def plot_losses(train_losses, validation_losses):
    epochs = range(1, len(train_losses) + 1)
    plt.figure(figsize=(7, 4))
    plt.plot(epochs, train_losses, marker="o", label="Entrenamiento")
    plt.plot(epochs, validation_losses, marker="o", label="Validación")
    plt.title("Pérdida por época")
    plt.xlabel("Época")
    plt.ylabel("Pérdida")
    plt.legend()
    plt.grid(True)
    plt.show()


def evaluate_model(network):
    """Returns the confusion matrix (rows: real class) and the metrics on the test set."""
    hp = network["hyperparameters"]
    _, _, test_loader = make_loaders(hp["classification"])
    classes = len(CLASS_NAMES[hp["classification"]])
    confusion = np.zeros((classes, classes), dtype=int)

    network["model"].eval()
    with torch.no_grad():
        for images, labels in test_loader:
            predictions = network["model"](images.to(DEVICE)).argmax(dim=1).cpu()
            for real, predicted in zip(labels.tolist(), predictions.tolist()):
                confusion[real, predicted] += 1

    hits = np.diag(confusion)
    precision = hits / np.maximum(confusion.sum(axis=0), 1)
    recall = hits / np.maximum(confusion.sum(axis=1), 1)

    return {
        "confusion": confusion,
        "accuracy": hits.sum() / confusion.sum(),
        "precision": precision,
        "recall": recall,
    }


def show_results(network, results):
    names = CLASS_NAMES[network["hyperparameters"]["classification"]]
    confusion = results["confusion"]

    print(f"Accuracy: {results['accuracy']:.4f}")
    print(f"Precisión (promedio): {results['precision'].mean():.4f}")
    print(f"Recall (promedio): {results['recall'].mean():.4f}\n")
    print(f"{'Clase':<14}{'Precisión':>10}{'Recall':>10}")
    for name, precision, recall in zip(names, results["precision"], results["recall"]):
        print(f"{name:<14}{precision:>10.4f}{recall:>10.4f}")

    plt.figure(figsize=(7, 6))
    plt.imshow(confusion, cmap="Blues")
    plt.title("Matriz de confusión")
    plt.xlabel("Predicción")
    plt.ylabel("Clase real")
    plt.xticks(range(len(names)), names, rotation=45, ha="right")
    plt.yticks(range(len(names)), names)
    for row in range(len(names)):
        for column in range(len(names)):
            color = "white" if confusion[row, column] > confusion.max() / 2 else "black"
            plt.text(column, row, confusion[row, column], ha="center", va="center", color=color)
    plt.colorbar()
    plt.tight_layout()
    plt.show()


def default_path(network):
    hp = network["hyperparameters"]
    return f"pytorch_fashion_{NOTEBOOK_NUMBER}_{hp['classification']}_{hp['architecture']}.pth"


def save_network(network, path):
    torch.save({"state_dict": network["model"].state_dict(),
                "hyperparameters": network["hyperparameters"]}, path)


def load_network(path):
    checkpoint = torch.load(path, map_location=DEVICE)
    hp = checkpoint["hyperparameters"]
    options = hp["optimizations"]

    model = build_model(hp["neurons_per_layer"], len(CLASS_NAMES[hp["classification"]]),
                        options["dropout"], options["batch_norm"])
    model.load_state_dict(checkpoint["state_dict"])
    return {"model": model, "hyperparameters": hp}


## Redes entregadas

Se compararon distintas cantidades de capas y neuronas usando la pérdida de validación, sin mirar el conjunto de prueba para decidir. Con diferencias muy pequeñas se prefirió la red con menos neuronas: por ejemplo, la rectangular de 10 clases con dos capas de 256 tiene una pérdida apenas mayor que las opciones más grandes (0.3011 frente a 0.2972). Todas usan Adam (`lr = 0.001`), *batch normalization*, *dropout* de 0.2 y *early stopping* con paciencia 4, con un máximo de 25 épocas.

| Red | Arquitectura | Épocas | Accuracy | Precisión | Recall |
|---|---|---|---|---|---|
| Rectangular, 10 clases | 784 → 256 → 256 → 10 | 18 | 89.18% | 89.22% | 89.18% |
| Convolucional, 10 clases | 784 → 526 → 268 → 10 | 12 | 88.50% | 88.61% | 88.50% |
| Rectangular, 4 clases | 784 → 128 → 4 | 16 | 97.47% | 97.71% | 96.79% |
| Convolucional, 4 clases | 784 → 394 → 4 | 13 | 97.42% | 97.24% | 97.11% |

En las redes convolucionales de 4 clases la interpolación llega hasta la capa de salida real (4 neuronas), para que cada capa disminuya en la misma cantidad.


## 5. Menú

La última celda debe dejar el menú corriendo (crear/cargar, entrenar, probar, guardar, salir).

In [5]:
def read_int(prompt, default, minimum=1):
    while True:
        text = input(f"{prompt} [{default}]: ").strip()
        if text == "":
            return default
        try:
            value = int(text)
        except ValueError:
            print("Escribe un número entero.")
            continue
        if value >= minimum:
            return value
        print(f"El valor debe ser al menos {minimum}.")


def read_float(prompt, default, minimum=0.0, maximum=float("inf")):
    while True:
        text = input(f"{prompt} [{default}]: ").strip()
        if text == "":
            return default
        try:
            value = float(text)
        except ValueError:
            print("Escribe un número.")
            continue
        if minimum <= value <= maximum:
            return value
        print(f"El valor debe estar entre {minimum} y {maximum}.")


def read_choice(prompt, choices, default):
    while True:
        text = input(f"{prompt} ({'/'.join(choices)}) [{default}]: ").strip().lower()
        if text == "":
            return default
        if text in choices:
            return text
        print("Opción no válida.")


def read_yes_no(prompt, default):
    answer = read_choice(prompt, ["s", "n"], "s" if default else "n")
    return answer == "s"


def read_training_options(options):
    """Optimizations that can change between training sessions."""
    options["optimizer"] = read_choice("Optimizador", list(OPTIMIZERS), options["optimizer"])
    options["learning_rate"] = read_float("Learning rate", options["learning_rate"], 1e-6)
    options["weight_decay"] = read_float("Weight decay", options["weight_decay"])
    options["early_stopping"] = read_yes_no("¿Usar early stopping?", options["early_stopping"])
    if options["early_stopping"]:
        options["patience"] = read_int("Paciencia (épocas)", options["patience"])


def menu_create():
    classification = read_choice("Clasificación", ["original", "simplified"], "original")
    architecture = read_choice("Tipo de red", ["rectangular", "convolutional"], "rectangular")
    hidden_layers = read_int("Número de capas ocultas", DEFAULT_HIDDEN_LAYERS)

    neurons_per_layer = DEFAULT_NEURONS
    if architecture == "rectangular":
        neurons_per_layer = read_int("Neuronas por capa", DEFAULT_NEURONS)

    options = dict(DEFAULT_OPTIONS)
    if read_yes_no("¿Configurar optimizaciones?", False):
        options["batch_norm"] = read_yes_no("¿Usar batch normalization?", False)
        options["dropout"] = read_float("Dropout (0 lo desactiva)", 0.0, 0.0, 0.9)
        read_training_options(options)

    network = create_network(classification, architecture, hidden_layers, neurons_per_layer,
                             **options)
    print("Red creada:", describe(network))
    return network


def menu_load():
    path = input("Archivo .pth: ").strip().strip('"')
    try:
        network = load_network(path)
    except (OSError, RuntimeError, KeyError, ValueError) as error:
        print("No se pudo cargar la red:", error)
        return None
    print("Red cargada:", describe(network))
    return network


def menu_save(network):
    default = default_path(network)
    path = input(f"Guardar en [{default}]: ").strip().strip('"') or default
    try:
        save_network(network, path)
    except OSError as error:
        print("No se pudo guardar la red:", error)
        return
    print("Red guardada en", path)


def run_menu():
    network = None

    while True:
        print("\n===== Fashion MNIST con PyTorch =====")
        print("Red actual:", describe(network) if network else "ninguna")
        print("1) Crear una red nueva")
        print("2) Cargar una red de un archivo")
        print("3) Entrenar")
        print("4) Probar")
        print("5) Guardar")
        print("6) Cambiar optimizaciones de entrenamiento")
        print("7) Salir")
        option = input("Opción: ").strip()

        if option == "1":
            network = menu_create()
        elif option == "2":
            network = menu_load() or network
        elif option == "7":
            print("Hasta luego.")
            return
        elif option not in ("3", "4", "5", "6"):
            print("Opción no válida.")
        elif network is None:
            print("Primero crea o carga una red.")
        elif option == "3":
            epochs = read_int("Número de épocas", DEFAULT_EPOCHS)
            plot_losses(*train_model(network, epochs))
        elif option == "4":
            show_results(network, evaluate_model(network))
        elif option == "5":
            menu_save(network)
        else:
            read_training_options(network["hyperparameters"]["optimizations"])


try:
    run_menu()
except (EOFError, NotImplementedError):
    # Without a keyboard (e.g. nbconvert) input() fails: stop here instead of crashing.
    print("\nNo hay entrada interactiva disponible: el menú se cierra.")



===== Fashion MNIST con PyTorch =====
Red actual: ninguna
1) Crear una red nueva
2) Cargar una red de un archivo
3) Entrenar
4) Probar
5) Guardar
6) Cambiar optimizaciones de entrenamiento
7) Salir

No hay entrada interactiva disponible: el menú se cierra.
